# 🔎 Prospector v3 — busca por raio

Escolha uma cidade e um raio em km. O sistema varre a área inteira, pega
também as cidades vizinhas, e devolve os comércios **sem site próprio**.

**Como usar:** clique no ▶️ de cada célula, de cima para baixo.

| Célula | O que faz |
|---|---|
| 1 | Você preenche cidade, raio e ramo — e monta o seu Drive |
| 2 | Carrega o motor da busca |
| 3 | Roda a busca, filtra e baixa as fotos |
| 4 | Monta a planilha e o PDF |
| 5 | Gera um site por lead, com a cor tirada da fachada |
| 6 | Monta o painel de prospecção |
| 7 | Abre o estúdio: você edita os sites conversando com o Claude |
| 8 | Baixa tudo num `.zip` |

---

### 💾 O trabalho não se perde mais

O Colab apaga o disco toda vez que desconecta. Antes, fechar a aba significava
perder fotos, sites e leads.

Agora a célula 1 monta o seu Google Drive e salva tudo em
**`Meu Drive / Prospector`**. Quando você voltar:

- Rode a célula 1 → ela reconhece a busca anterior
- Rode a célula 3 → ela **retoma de onde parou, sem gastar consulta na API**

Só quando você quiser buscar de novo do zero, ligue `REFAZER_BUSCA` na célula 1.

> O Colab vai pedir permissão para acessar o Drive na primeira vez. É normal.

---

### 🔑 As duas chaves

As chaves **não ficam escritas no notebook**. Assim elas não vão junto quando
você compartilha o arquivo.

**A do Google** fica no cofre do Colab. No menu da esquerda, clique no 🔑:

| Nome do secret | Onde pegar |
|---|---|
| `GOOGLE_PLACES_API_KEY` | console.cloud.google.com |

Ligue a chavinha **"Acesso ao notebook"**.

**A do Claude** você cola dentro do próprio estúdio, no botão **"Chave"**.
Ela fica guardada só no seu navegador — não entra no `.zip`, então você pode
mandar a pasta para o cliente sem medo.

---

⚠️ **Sobre as fotos:** as imagens do perfil pertencem a quem as enviou ao Google.
Use para montar a proposta e mostrar o preview para o dono. Só publique em
site aberto depois do sim dele.

In [ ]:
#@title ⚙️ 1. Preencha aqui e clique no ▶️

# A chave NAO fica escrita aqui. Ela fica no cofre do Colab:
# menu da esquerda -> 🔑 -> "Adicionar novo secret"
#   Nome:  GOOGLE_PLACES_API_KEY
#   Valor: sua chave
#   E ligue a chavinha "Acesso ao notebook"
# Assim a chave nao vai junto quando voce compartilha ou salva o notebook.

CIDADE_CENTRO = "Sumaré" #@param {type:"string"}
ESTADO = "SP" #@param {type:"string"}
RAIO_KM = 20 #@param {type:"slider", min:5, max:100, step:5}
RAMOS = "pet shop" #@param {type:"string"}
NOTA_MINIMA = 4 #@param {type:"slider", min:3, max:5, step:0.1}
MIN_AVALIACOES = 20 #@param {type:"slider", min:0, max:200, step:5}
BAIXAR_FOTOS = True #@param {type:"boolean"}
MAX_FOTOS_POR_LOCAL = 8 #@param {type:"slider", min:1, max:10, step:1}
PAGINAS_POR_PONTO = 2 #@param {type:"slider", min:1, max:3, step:1}
LIMITE_CONSULTAS = 250 #@param {type:"slider", min:50, max:600, step:50}

# Salvar no Drive faz o trabalho sobreviver quando o Colab desconecta.
# Sem isso, tudo (fotos, sites, leads) some ao fechar a aba.
SALVAR_NO_DRIVE = True #@param {type:"boolean"}
# Ligue so quando quiser buscar de novo do zero e gastar consulta na API.
REFAZER_BUSCA = False #@param {type:"boolean"}

API_KEY, _PROBLEMA = "", ""
try:
    from google.colab import userdata
    API_KEY = (userdata.get("GOOGLE_PLACES_API_KEY") or "").strip()
    if not API_KEY:
        _PROBLEMA = "vazio"
except ImportError:
    _PROBLEMA = "fora-do-colab"
except Exception as _e:
    _n = type(_e).__name__
    _PROBLEMA = ("nao-existe" if "NotFound" in _n
                 else "chavinha-desligada" if "Access" in _n
                 else "sem-permissao" if "Timeout" in _n
                 else f"outro:{_n}")

CONFIG = {
    "cidade": CIDADE_CENTRO.strip(),
    "estado": ESTADO.strip(),
    "raio_km": float(RAIO_KM),
    "sub_km": 20.0,
    "ramos": [r.strip() for r in RAMOS.split(",") if r.strip()],
    "nota_minima": float(NOTA_MINIMA),
    "min_avaliacoes": int(MIN_AVALIACOES),
    "baixar_fotos": BAIXAR_FOTOS,
    "max_fotos_por_local": int(MAX_FOTOS_POR_LOCAL),
    "paginas": int(PAGINAS_POR_PONTO),
    "limite": int(LIMITE_CONSULTAS),
    "largura_foto": 1600,
    "pasta_saida": "resultado",
    "idioma": "pt-BR",
}

import re as _re, unicodedata as _ud
from pathlib import Path as _Path

def _apelido(t):
    t = _ud.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (_re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:60] or "sem-nome"

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = _Path("/content/drive/MyDrive/Prospector")
else:
    RAIZ = _Path("resultado")

CONFIG["pasta_saida"] = str(RAIZ)
base = RAIZ / _apelido(f"{CONFIG['cidade']}-{CONFIG['estado']}-{int(CONFIG['raio_km'])}km")
base.mkdir(parents=True, exist_ok=True)
ANTERIOR = base / "leads.json"

print(f"Centro: {CONFIG['cidade']}/{CONFIG['estado']} — raio de {int(CONFIG['raio_km'])} km")
print(f"Ramos: {', '.join(CONFIG['ramos'])}")
print(f"Filtro: nota >= {CONFIG['nota_minima']} | avaliações >= {CONFIG['min_avaliacoes']} | sem site próprio")

if API_KEY:
    print(f"\n✓ Chave do Google: ok (termina em ...{API_KEY[-4:]})")
else:
    print("\n" + "═" * 58)
    print("⚠️  SEM A CHAVE, A BUSCA NÃO VAI RODAR.")
    print("═" * 58)
    if _PROBLEMA == "nao-existe":
        print("O secret não existe com esse nome.")
        print("\nNo menu da esquerda, clique no \U0001f511 e crie um com o nome")
        print("EXATAMENTE assim (maiúsculas e underline, sem espaço):")
        print("\n    GOOGLE_PLACES_API_KEY\n")
    elif _PROBLEMA == "chavinha-desligada":
        print("O secret existe, mas está bloqueado para este notebook.")
        print("\nClique no \U0001f511, ache GOOGLE_PLACES_API_KEY e ligue a")
        print("chavinha azul 'Acesso ao notebook' do lado dele.\n")
    elif _PROBLEMA == "vazio":
        print("O secret existe mas o campo 'Valor' está em branco.")
        print("Clique no \U0001f511 e cole a chave no campo Valor.\n")
    elif _PROBLEMA == "sem-permissao":
        print("Este notebook ainda não tem permissão para ler o cofre.")
        print("\nCada cópia do notebook precisa da sua própria autorização,")
        print("e numa cópia nova ela começa desligada.")
        print("\nClique no \U0001f511 do menu, ache GOOGLE_PLACES_API_KEY e ligue")
        print("a chavinha azul 'Acesso ao notebook'. Se aparecer um aviso")
        print("pedindo permissão, clique em Permitir.\n")
    elif _PROBLEMA == "fora-do-colab":
        print("Isto não está rodando no Google Colab.\n")
    else:
        print(f"Erro inesperado ao ler o cofre: {_PROBLEMA}\n")
    print("Depois de arrumar, rode ESTA célula de novo antes de seguir.")
    print("═" * 58)

print(f"\nPasta de trabalho: {base}")
if SALVAR_NO_DRIVE:
    print("Salvando no seu Drive — o trabalho continua aí quando você fechar e voltar.")
else:
    print("⚠️  Sem o Drive, tudo se perde quando o Colab desconectar.")

if ANTERIOR.exists() and not REFAZER_BUSCA:
    import json as _json
    _n = len(_json.loads(ANTERIOR.read_text(encoding="utf-8")))
    print(f"\n✓ Achei uma busca anterior com {_n} leads.")
    print("  A célula 3 vai retomar dela, sem gastar consulta na API.")
    print("  Para buscar de novo do zero, ligue REFAZER_BUSCA aqui em cima.")

In [ ]:
#@title 🔧 2. Motor (só clique no ▶️)

import csv, json, math, re, time, unicodedata
from pathlib import Path
import requests

BASE = "https://places.googleapis.com/v1/places:searchText"

FIELDS = ",".join([
    "places.id", "places.displayName", "places.formattedAddress",
    "places.nationalPhoneNumber", "places.rating", "places.userRatingCount",
    "places.websiteUri", "places.googleMapsUri", "places.location",
    "places.primaryTypeDisplayName", "places.businessStatus", "places.photos",
    "nextPageToken",
])

SOCIAIS = (
    "instagram.com", "facebook.com", "fb.com", "linktr.ee", "linktree",
    "wa.me", "api.whatsapp.com", "whatsapp.com", "linkbio", "beacons.ai",
    "bio.link", "youtube.com", "tiktok.com", "ifood.com", "linkedin.com",
    "google.com", "sites.google.com", "business.site", "negocio.site",
)

def slug(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:60] or "sem-nome"

def tem_site_proprio(url):
    if not url:
        return False
    u = url.lower()
    return not any(d in u for d in SOCIAIS)

# Caminhos do Instagram que NAO sao perfil de ninguem
NAO_PERFIL = {"invites", "explore", "accounts", "p", "reel", "reels",
              "stories", "direct", "about", "developer", "legal"}

def achar_instagram(p):
    """Pega o @ do perfil quando o link cadastrado e um Instagram.

    Muito comercio cadastra o link de convite (instagram.com/invites/contact/)
    em vez do perfil. Isso nao e um @ de ninguem, entao devolve vazio em vez
    de inventar um arroba quebrado que iria parar no site do lead."""
    url = (p.get("websiteUri") or "").lower()
    if "instagram.com" not in url:
        return ""
    h = url.rstrip("/").split("instagram.com/")[-1].split("?")[0].strip("/")
    if not h or "/" in h or h in NAO_PERFIL:
        return ""
    if not re.fullmatch(r"[a-z0-9._]{1,30}", h):
        return ""
    return "@" + h

def distancia_km(la1, lo1, la2, lo2):
    R = 6371.0
    p1, p2 = math.radians(la1), math.radians(la2)
    dp, dl = p2 - p1, math.radians(lo2 - lo1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def geocodificar(cidade, estado):
    h = {"Content-Type": "application/json", "X-Goog-Api-Key": API_KEY,
         "X-Goog-FieldMask": "places.location,places.formattedAddress"}
    b = {"textQuery": f"{cidade}, {estado}, Brasil", "languageCode": "pt-BR", "pageSize": 1}
    r = requests.post(BASE, headers=h, json=b, timeout=30)
    if r.status_code != 200:
        print(f"Erro ao localizar a cidade: {r.status_code} \u2014 {r.text[:200]}")
        return None
    places = r.json().get("places", [])
    if not places:
        return None
    loc = places[0]["location"]
    return loc["latitude"], loc["longitude"], places[0].get("formattedAddress", "")

def gerar_pontos(lat0, lng0, raio_km, sub_km):
    passo = sub_km * 1.4
    n = int(math.ceil((raio_km + sub_km) / passo))
    pontos = []
    for i in range(-n, n + 1):
        for j in range(-n, n + 1):
            dy, dx = i * passo, j * passo
            if math.hypot(dx, dy) > raio_km + sub_km * 0.7:
                continue
            lat = lat0 + dy / 110.574
            lng = lng0 + dx / (111.320 * math.cos(math.radians(lat0)))
            pontos.append((lat, lng))
    return pontos

def buscar_ponto(ramo, lat, lng, cfg, contador):
    h = {"Content-Type": "application/json", "X-Goog-Api-Key": API_KEY,
         "X-Goog-FieldMask": FIELDS}
    # locationBias aceita c\u00edrculo; locationRestriction no Text Search s\u00f3 aceita ret\u00e2ngulo
    circulo = {"circle": {"center": {"latitude": lat, "longitude": lng},
                          "radius": cfg["sub_km"] * 1000}}
    out, token, pag = [], None, 0
    while pag < cfg["paginas"]:
        if contador["n"] >= cfg["limite"]:
            return out
        b = {"textQuery": ramo, "languageCode": cfg["idioma"], "pageSize": 20,
             "locationBias": circulo}
        if token:
            b["pageToken"] = token
        r = requests.post(BASE, headers=h, json=b, timeout=30)
        contador["n"] += 1
        if r.status_code != 200:
            print(f"    ! {r.status_code}: {r.text[:200]}")
            break
        d = r.json()
        out.extend(d.get("places", []))
        token = d.get("nextPageToken")
        pag += 1
        if not token:
            break
        time.sleep(2)
    return out

def baixar_fotos(place, destino, cfg):
    fotos = place.get("photos", [])[: cfg["max_fotos_por_local"]]
    if not fotos:
        return []
    destino.mkdir(parents=True, exist_ok=True)
    salvas = []
    for i, f in enumerate(fotos, 1):
        nome = f.get("name")
        if not nome:
            continue
        url = f"https://places.googleapis.com/v1/{nome}/media?maxWidthPx={cfg['largura_foto']}&key={API_KEY}"
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code != 200:
                continue
            arq = destino / f"{i:02d}.jpg"
            arq.write_bytes(resp.content)
            salvas.append(str(arq))
            autores = [a.get("displayName", "") for a in f.get("authorAttributions", [])]
            if autores:
                with open(destino / "creditos.txt", "a", encoding="utf-8") as fh:
                    fh.write(f"{arq.name}: {', '.join(autores)}\n")
        except requests.RequestException:
            continue
    return salvas

print("\u2705 Motor carregado.")

In [ ]:
#@title ▶️ 3. Rodar a busca

import json as _json

if not API_KEY:
    print("⚠️  A célula 1 não achou a chave do Google.")
    print("    Volte nela, siga o que ela mandou fazer, rode ela de novo")
    print("    e só então rode esta aqui. Nada foi buscado.")
    leads = []

elif ANTERIOR.exists() and not REFAZER_BUSCA:
    # Retoma o trabalho salvo no Drive em vez de buscar tudo de novo.
    leads = _json.loads(ANTERIOR.read_text(encoding="utf-8"))
    print(f"Retomando a busca salva: {len(leads)} leads.")
    print("Nenhuma consulta gasta na API do Google.")
    print("Para buscar de novo, ligue REFAZER_BUSCA na célula 1 e rode as duas de novo.\n")
    try:
        import pandas as pd
        display(pd.DataFrame(leads)[["nome", "telefone", "nota", "avaliacoes",
                                     "km_do_centro", "instagram", "qtd_fotos"]])
    except Exception:
        pass

else:
    cfg = CONFIG
    centro = geocodificar(cfg["cidade"], cfg["estado"])
    if not centro:
        print("\n⚠️  Não consegui localizar a cidade.")
        print("    Se o erro acima foi 403, a chave não está sendo aceita:")
        print("    confira se a Places API (New) está ativada para ela no")
        print("    Google Cloud, em APIs e serviços → APIs ativadas.")
        print("    Se foi outro erro, confira a cidade e a sigla do estado.")
        leads = []

    if centro:
        lat0, lng0, endereco_centro = centro
        pontos = gerar_pontos(lat0, lng0, cfg["raio_km"], cfg["sub_km"])
        previsto = len(pontos) * len(cfg["ramos"]) * cfg["paginas"]

        print(f"Centro: {endereco_centro}")
        print(f"{len(pontos)} pontos de varredura para cobrir {int(cfg['raio_km'])} km")
        print(f"At\u00e9 {previsto} consultas (teto de seguran\u00e7a: {cfg['limite']})\n")

        base = Path(cfg["pasta_saida"]) / slug(f"{cfg['cidade']}-{cfg['estado']}-{int(cfg['raio_km'])}km")
        base.mkdir(parents=True, exist_ok=True)

        contador = {"n": 0}
        vistos, brutos = set(), []
        erros = 0

        for ramo in cfg["ramos"]:
            print(f"> {ramo}")
            for idx, (la, lo) in enumerate(pontos, 1):
                if contador["n"] >= cfg["limite"]:
                    print("  (teto de consultas atingido \u2014 parando)")
                    break
                achados = buscar_ponto(ramo, la, lo, cfg, contador)
                if not achados:
                    erros += 1
                novos = 0
                for p in achados:
                    pid = p.get("id")
                    if pid and pid not in vistos:
                        vistos.add(pid)
                        p["_ramo"] = ramo
                        brutos.append(p)
                        novos += 1
                print(f"  ponto {idx}/{len(pontos)} \u2014 +{novos} novos (total {len(brutos)})")
                if erros >= 3 and not brutos:
                    print("\n  \u26a0\ufe0f V\u00e1rias buscas seguidas sem retorno \u2014 parando para n\u00e3o gastar \u00e0 toa.")
                    break

        print(f"\n{contador['n']} consultas feitas. Filtrando {len(brutos)} estabelecimentos...\n")

        leads = []
        for p in brutos:
            nota = p.get("rating") or 0
            n_aval = p.get("userRatingCount") or 0
            if nota < cfg["nota_minima"] or n_aval < cfg["min_avaliacoes"]:
                continue
            if tem_site_proprio(p.get("websiteUri")):
                continue
            if p.get("businessStatus") not in (None, "OPERATIONAL"):
                continue

            loc = p.get("location", {})
            dist = distancia_km(lat0, lng0, loc.get("latitude", lat0), loc.get("longitude", lng0))
            if dist > cfg["raio_km"]:
                continue

            nome = p.get("displayName", {}).get("text", "")
            pasta = base / "fotos" / f"{slug(nome)}-{p['id'][-6:]}"
            fotos = baixar_fotos(p, pasta, cfg) if cfg["baixar_fotos"] else []

            leads.append({
                "nome": nome,
                "telefone": p.get("nationalPhoneNumber", ""),
                "nota": nota,
                "avaliacoes": n_aval,
                "instagram": achar_instagram(p),
                "km_do_centro": round(dist, 1),
                "link_cadastrado": p.get("websiteUri", "") or "",
                "ramo_busca": p.get("_ramo", ""),
                "categoria_google": p.get("primaryTypeDisplayName", {}).get("text", ""),
                "endereco": p.get("formattedAddress", ""),
                "maps": p.get("googleMapsUri", ""),
                "qtd_fotos": len(fotos),
                "pasta_fotos": str(pasta) if fotos else "",
                "place_id": p.get("id", ""),
            })
            print(f"  \u2713 {nome} \u2014 {nota}\u2605 ({n_aval}) \u2014 {dist:.0f} km \u2014 {len(fotos)} fotos")

        if leads:
            leads.sort(key=lambda x: (-x["avaliacoes"], -x["nota"]))
            with open(base / "leads.csv", "w", newline="", encoding="utf-8-sig") as f:
                w = csv.DictWriter(f, fieldnames=list(leads[0].keys()))
                w.writeheader()
                w.writerows(leads)
            (base / "leads.json").write_text(json.dumps(leads, ensure_ascii=False, indent=2), encoding="utf-8")

            print(f"\n{'-'*45}")
            print(f"{len(leads)} leads sem site em {int(cfg['raio_km'])} km de {cfg['cidade']}")
            print(f"{sum(1 for l in leads if l['telefone'])} com telefone | {sum(1 for l in leads if l['instagram'])} com Instagram")
            try:
                import pandas as pd
                display(pd.DataFrame(leads)[["nome", "telefone", "nota", "avaliacoes", "km_do_centro", "instagram", "qtd_fotos"]])
            except Exception:
                pass
        elif brutos:
            print("\nAchei estabelecimentos, mas nenhum passou nos filtros.")
            print("Baixe a nota m\u00ednima ou o m\u00ednimo de avalia\u00e7\u00f5es e rode de novo.")
        else:
            print("\nA busca n\u00e3o retornou nada. Veja as mensagens de erro acima.")

In [ ]:
#@title 📄 Planilha e PDF

!pip install -q fpdf2
from fpdf import FPDF
import csv

def limpo(s):
    return str(s).encode("latin-1", "ignore").decode("latin-1")

# ---------- CSV que abre certo no Excel brasileiro ----------
with open(base / "leads_excel.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["Nome", "Telefone", "Endereco", "Nota", "Avaliacoes", "Instagram", "Km"])
    for l in leads:
        w.writerow([l["nome"], l["telefone"], l["endereco"],
                    str(l["nota"]).replace(".", ","), l["avaliacoes"],
                    l["instagram"], str(l["km_do_centro"]).replace(".", ",")])

# ---------- PDF em tabela ----------
MARGEM = 10
COLS = [
    ("N",         8),
    ("NOME",     58),
    ("TELEFONE", 34),
    ("ENDERECO", 118),
    ("NOTA",     14),
    ("AVAL.",    16),
    ("KM",       12),
]
LARGURA = sum(c[1] for c in COLS)   # 260 mm, cabe em A4 deitado

pdf = FPDF(orientation="L", format="A4")
pdf.set_auto_page_break(auto=False)
pdf.set_margins(MARGEM, MARGEM, MARGEM)

def corta(texto, larg):
    t = limpo(texto)
    if pdf.get_string_width(t) <= larg - 3:
        return t
    while len(t) > 3 and pdf.get_string_width(t + "..") > larg - 3:
        t = t[:-1]
    return t + ".."

def cabecalho():
    pdf.add_page()
    pdf.set_xy(MARGEM, MARGEM)
    pdf.set_font("helvetica", "B", 14)
    pdf.set_text_color(0, 0, 0)
    pdf.cell(LARGURA, 8, limpo(f"Leads sem site - {CONFIG['cidade']}/{CONFIG['estado']}"),
             new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("helvetica", "", 9)
    pdf.set_text_color(110, 110, 110)
    pdf.cell(LARGURA, 5, limpo(f"{len(leads)} estabelecimentos - raio de {int(CONFIG['raio_km'])} km"),
             new_x="LMARGIN", new_y="NEXT")
    pdf.ln(3)

    pdf.set_font("helvetica", "B", 8)
    pdf.set_fill_color(45, 45, 45)
    pdf.set_text_color(255, 255, 255)
    for titulo, larg in COLS:
        pdf.cell(larg, 7, limpo(titulo), align="L", fill=True)
    pdf.ln(7)
    pdf.set_text_color(0, 0, 0)

cabecalho()

for i, l in enumerate(leads, 1):
    if pdf.get_y() > 180:
        cabecalho()

    y = pdf.get_y()
    if i % 2 == 0:
        pdf.set_fill_color(246, 246, 246)
        pdf.rect(MARGEM, y, LARGURA, 7, style="F")

    pdf.set_xy(MARGEM, y)
    valores = [
        str(i),
        l["nome"],
        l["telefone"] or "-",
        l["endereco"],
        str(l["nota"]).replace(".", ","),
        str(l["avaliacoes"]),
        str(l["km_do_centro"]).replace(".", ","),
    ]
    for (titulo, larg), valor in zip(COLS, valores):
        pdf.set_font("helvetica", "B" if titulo == "NOME" else "", 8)
        pdf.cell(larg, 7, corta(valor, larg), align="L")
    pdf.ln(7)

    pdf.set_draw_color(225, 225, 225)
    pdf.line(MARGEM, pdf.get_y(), MARGEM + LARGURA, pdf.get_y())

caminho = str(base / "leads.pdf")
pdf.output(caminho)
print(f"PDF gerado com {len(leads)} leads.")
print("Sai no .zip da última célula: leads.pdf e leads_excel.csv")

In [ ]:
#@title 🎨 Gerador de sites — a cor sai da fachada, do logo ou da paleta do nicho

#@markdown O site nasce com horário, formas de pagamento e um parágrafo de
#@markdown apresentação já preenchidos, para não parecer vazio na hora de
#@markdown mostrar ao dono. São textos genéricos, marcados com "a confirmar".
#@markdown Depois de fechar, você troca pelos reais no estúdio (célula 7) e
#@markdown desliga a marcação aqui.
MARCAR_A_CONFIRMAR = True #@param {type:"boolean"}

import json, shutil, re, unicodedata, colorsys
from pathlib import Path
from string import Template
from html import escape as esc
from PIL import Image
from google.colab import files

# ─────────── cor de reserva de cada nicho ───────────
# Usada quando a foto nao entrega cor propria (fachada branca, foto escura,
# so o interior da loja). O resto da paleta e sempre derivado dela.
NICHO = {
    "pet":     "#3E8E5A",
    "oficina": "#C24E12",
    "clinica": "#1E6FA8",
    "comida":  "#A32B33",
    "beleza":  "#7A3E86",
    "escola":  "#1F6E63",
    "padrao":  "#2B5F8C",
}

SERVICOS = {
    "pet": [("Banho e tosa", "Banho, tosa, unhas e limpeza de ouvido. Atende do chihuahua ao rottweiler."),
            ("Rações e acessórios", "As rações que o bicho daqui já come. Se o seu come outra, a gente traz."),
            ("Atendimento próximo", "A gente fica aqui do lado. Seu pet é cliente, não número.")],
    "oficina": [("Diagnóstico honesto", "A gente mostra o que precisa fazer. Ninguém sai daqui sem entender o orçamento."),
                ("Manutenção preventiva", "Revisão, óleo, filtro, freios. A gente avisa quando está na hora de mexer."),
                ("Prazo que se cumpre", "Se a gente disser que fica pronto segunda, segunda fica. Sem surpresa.")],
    "clinica": [("Atendimento humano", "Consulta de verdade. A gente escuta e examina. Não é correria."),
                ("Estrutura completa", "Consultório limpo e equipado. A gente esteriliza tudo direitinho."),
                ("Agendamento fácil", "Marca por telefone e é atendido no horário. Sem demora.")],
    "comida": [("Feito na hora", "Nada congelado. A gente faz na hora em que você chega."),
               ("Receita da casa", "É a mesma receita desde o começo. Ninguém copiou."),
               ("Salão e retirada", "Tem mesa para comer aqui ou você leva para casa.")],
    "beleza": [("Profissionais formados", "Todo mundo aqui fez curso. A mão é leve e experiente."),
               ("Produtos de linha", "A gente usa produto profissional, não o barato de supermercado."),
               ("Seu horário respeitado", "Você marca a hora e a gente está pronto. Nada de atraso.")],
    "escola": [("Turmas pequenas", "Poucos alunos por sala. O professor sabe o nome de cada um."),
               ("Equipe que fica", "Os professores ficam anos aqui. Conhecem as famílias."),
               ("Portas abertas", "Venha conhecer a escola antes de decidir. Sem hora marcada.")],
    "padrao": [("Atendimento próximo", "A gente é daqui. Conhece o bairro e conhece você."),
               ("Qualidade comprovada", "A gente sobrevive porque faz certo. Reputação não se compra."),
               ("Fácil de encontrar", "A gente está aqui do lado. Perto de casa, fácil de achar.")],
}

# Textos de rascunho: genericos de proposito, para o site parecer completo
# na hora da conversa. Nada aqui e um dado real do comercio — por isso vao
# marcados com "a confirmar" ate o dono passar os certos.
RASCUNHO = {
    "pet":     ("Cuidar de bicho é o que a gente faz o dia inteiro, aqui no bairro. "
                "Quem traz uma vez costuma voltar — e trazer o vizinho junto.",
                "Segunda a sexta, 8h às 18h · Sábado, 8h às 13h"),
    "oficina": ("Carro conhecido, mecânico conhecido. A gente atende quem mora aqui "
                "por perto e faz questão de explicar o que vai mexer, antes de mexer.",
                "Segunda a sexta, 8h às 18h · Sábado, 8h às 12h"),
    "clinica": ("Atendimento de gente para gente, sem fila e sem correria. "
                "Você marca, é atendido no horário e sai daqui entendendo o que tem.",
                "Segunda a sexta, 8h às 19h · Sábado, 8h às 12h"),
    "comida":  ("Comida feita na hora, do jeito que a casa faz desde o começo. "
                "Vem gente de longe, mas quem mora aqui perto é quem sustenta.",
                "Terça a domingo, 11h às 15h e 18h às 23h"),
    "beleza":  ("Aqui você senta na cadeira e sai do jeito que combinou. "
                "Profissional formado, produto de linha e horário respeitado.",
                "Terça a sábado, 9h às 19h"),
    "escola":  ("Turma pequena, professor que fica e porta aberta para a família. "
                "Venha conhecer antes de decidir — a gente prefere assim.",
                "Segunda a sexta, 7h às 18h"),
    "padrao":  ("A gente é daqui e atende quem é daqui. Conhece o bairro, "
                "conhece o cliente e faz questão de manter a fama que construiu.",
                "Segunda a sexta, 8h às 18h · Sábado, 8h às 13h"),
}

PAGAMENTO = "Dinheiro, Pix, débito e crédito"

def familia(t):
    r = t.lower()
    if any(k in r for k in ("pet", "veterin", "animal", "agro")): return "pet"
    if any(k in r for k in ("oficina", "mecan", "auto", "funilar", "pneu")): return "oficina"
    if any(k in r for k in ("clinic", "odonto", "dent", "saude", "fisio", "medic")): return "clinica"
    if any(k in r for k in ("restaur", "pizz", "lanch", "bar", "cafe", "padar", "espet")): return "comida"
    if any(k in r for k in ("salao", "barbear", "estetic", "cabelo", "unha", "spa")): return "beleza"
    if any(k in r for k in ("escola", "colegio", "creche", "infantil", "ensino", "curso")): return "escola"
    return "padrao"

def sl(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:50] or "site"

def contatos(tel):
    d = re.sub(r"\D", "", tel or "")
    if d and not d.startswith("55"): d = "55" + d
    cel = len(d) == 13 and d[4] == "9"
    return ("tel:+" + d if d else ""), ("https://wa.me/" + d if cel else "")

# ─────────── achar a cor da marca ───────────

def _rgb(h, s, l):
    r, g, b = colorsys.hls_to_rgb(h % 1.0, l, s)
    return (r * 255, g * 255, b * 255)

def _hex(rgb):
    return "#%02X%02X%02X" % tuple(max(0, min(255, round(c))) for c in rgb)

def _luz_percebida(rgb):
    def canal(c):
        c = c / 255
        return c / 12.92 if c <= 0.03928 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = (canal(c) for c in rgb)
    return 0.2126 * r + 0.7152 * g + 0.0722 * b

def _contraste(a, b):
    la, lb = _luz_percebida(a), _luz_percebida(b)
    return (max(la, lb) + 0.05) / (min(la, lb) + 0.05)

def _limiar_sat(graus):
    """Quanto a cor precisa ser viva para contar como cor de marca.

    A faixa de 20 a 50 graus e madeira, pelo de bicho, papelao, piso e pele.
    Isso aparece em toda foto de loja e nao diz nada sobre a marca, entao ali
    so aceito se for bem viva mesmo — um toldo laranja de verdade passa, a
    prateleira de madeira nao."""
    return 0.55 if 20 <= graus <= 50 else 0.42

def paleta_das_fotos(paths, reserva):
    """Procura nas primeiras fotos a cor que a loja usa de verdade: toldo,
    placa, fachada pintada, logo. Devolve (hex, origem), com origem 'foto'
    quando achou cor propria e 'nicho' quando caiu na cor de reserva.

    Tres coisas precisam ser verdade para virar 'foto':
    a cor tem de ser viva o bastante para o seu tom, tem de ocupar uma parte
    real da imagem, e tem de estar concentrada num tom so. Um toldo cumpre as
    tres; uma prateleira cheia de produtos coloridos falha na ultima, e um
    cachorro caramelo falha na primeira."""
    faixas, aproveitados, olhados = {}, 0, 0
    for peso, p in zip((3, 2, 1, 1), paths[:4]):
        try:
            im = Image.open(p).convert("RGB")
        except Exception:
            continue
        im.thumbnail((150, 150))
        larg, alt = im.size
        im = im.crop((0, int(alt * 0.15), larg, alt))   # a faixa de cima quase sempre e ceu
        for r, g, b in im.getdata():
            olhados += 1
            h, l, s = colorsys.rgb_to_hls(r / 255, g / 255, b / 255)
            graus = h * 360
            if s < _limiar_sat(graus) or not (0.20 <= l <= 0.80):
                continue
            aproveitados += 1
            f = int(graus // 12)
            acc = faixas.setdefault(f, [0.0, 0.0, 0])
            acc[0] += peso * s
            acc[1] += s
            acc[2] += 1

    if not faixas or not olhados:
        return reserva, "nicho"
    if aproveitados < olhados * 0.06:          # respingo de cor nao e marca
        return reserva, "nicho"

    f = max(faixas, key=lambda k: faixas[k][0])
    _, soma_sat, quantos = faixas[f]
    if quantos < aproveitados * 0.35:          # cor espalhada nao e marca
        return reserva, "nicho"

    h = (f + 0.5) / 30
    return _hex(_rgb(h, min(0.70, max(0.34, soma_sat / quantos)), 0.45)), "foto"

def derivar(destaque):
    """Todas as cores do site saem do mesmo matiz — por isso combinam entre si.
    As luzes sao fixas e o destaque e escurecido ate passar em 4.5:1 com texto
    branco, entao nenhum site sai com botao ilegivel."""
    r, g, b = (int(destaque[i:i+2], 16) / 255 for i in (1, 3, 5))
    h, _, s = colorsys.rgb_to_hls(r, g, b)
    vivo = min(0.66, max(0.36, s))
    luz = 0.44
    forte = _rgb(h, vivo, luz)
    while _contraste(forte, (255, 255, 255)) < 4.5 and luz > 0.16:
        luz -= 0.02
        forte = _rgb(h, vivo, luz)
    tinta = _rgb(h, 0.22, 0.11)
    return {
        "destaque":       _hex(forte),                   # botoes e detalhes
        "destaque_claro": _hex(_rgb(h, vivo, 0.70)),     # o destaque sobre a faixa escura
        "tinta":          _hex(tinta),                   # quase preto, com o matiz da marca
        "tinta_rgb":      ", ".join(str(round(c)) for c in tinta),
        "fundo":          _hex(_rgb(h, 0.14, 0.972)),    # quase branco, mesmo matiz
        "suave":          _hex(_rgb(h, 0.12, 0.915)),    # bordas e espera de imagem
    }

# ─────────── template do site ───────────
SITE = Template("""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>$nome — $cidade</title>
<meta name="description" content="$nome em $cidade. Nota $nota no Google, $aval avaliações. $endereco">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{box-sizing:border-box}
body,h1,h2,h3,p,ol,li,figure,dl,dd{margin:0;padding:0}
:root{
  --tinta:$tinta; --tinta-rgb:$tinta_rgb; --fundo:$fundo;
  --destaque:$destaque; --destaque-claro:$destaque_claro; --suave:$suave;
  --carta:#fff;
  --linha:rgba(var(--tinta-rgb),.13);
  --sutil:rgba(var(--tinta-rgb),.62);
  --tec:"Space Grotesk",system-ui,sans-serif;
  --txt:"Inter",system-ui,-apple-system,sans-serif;
  --env:min(1120px,100% - 44px);
  --raio:10px;
}
html{scroll-behavior:smooth;-webkit-text-size-adjust:100%}
body{background:var(--fundo);color:var(--tinta);font-family:var(--txt);font-size:16px;
  line-height:1.6;-webkit-font-smoothing:antialiased}
.env{width:var(--env);margin-inline:auto}
img{display:block;max-width:100%}
a{color:inherit}
:focus-visible{outline:2px solid var(--destaque);outline-offset:3px;border-radius:4px}
.rotulo{font-size:.72rem;font-weight:600;letter-spacing:.12em;
  text-transform:uppercase;color:var(--destaque)}

/* topo */
.topo{position:sticky;top:0;z-index:40;background:rgba(255,255,255,.85);
  backdrop-filter:saturate(150%) blur(14px);border-bottom:1px solid var(--linha)}
.topo .env{display:flex;align-items:center;justify-content:space-between;gap:20px;height:64px}
.marca{font-family:var(--tec);font-weight:700;font-size:1.02rem;letter-spacing:-.02em;
  white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.acoes{display:flex;align-items:center;gap:14px;flex-shrink:0}
.link-tel{font-weight:600;font-size:.9rem;text-decoration:none;white-space:nowrap}
.pilula{display:inline-flex;align-items:center;background:var(--destaque);color:#fff;
  text-decoration:none;font-weight:600;font-size:.86rem;padding:10px 18px;border-radius:100px;
  white-space:nowrap;transition:filter .18s,transform .18s}
.pilula:hover{filter:brightness(1.09);transform:translateY(-1px)}

/* abertura */
.capa{width:100%;aspect-ratio:16/7;max-height:56vh;object-fit:cover;background:var(--suave)}
.abertura{padding:46px 0 0}
h1{font-family:var(--tec);font-size:clamp(1.9rem,4.4vw,3rem);line-height:1.1;
  letter-spacing:-.03em;font-weight:700;margin-top:12px}
.onde{margin-top:13px;color:var(--sutil);font-size:1.02rem;max-width:46ch}
.selo{display:inline-flex;align-items:center;gap:8px;margin-top:22px;padding:8px 16px;
  border:1px solid var(--linha);border-radius:100px;background:var(--carta);font-size:.9rem}
.selo b{font-family:var(--tec);font-weight:700}
.estrela{color:var(--destaque)}

/* numeros */
.numeros{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:1px;
  background:var(--linha);border:1px solid var(--linha);border-radius:var(--raio);
  overflow:hidden;margin-top:46px}
.numeros>div{background:var(--carta);padding:20px 22px}
.numeros dt{font-size:.68rem;font-weight:600;letter-spacing:.1em;
  text-transform:uppercase;color:var(--sutil)}
.numeros dd{font-family:var(--tec);margin-top:6px;font-size:1.35rem;
  font-weight:700;letter-spacing:-.02em}

/* mosaico */
.mosaico{display:grid;grid-template-columns:repeat(4,1fr);gap:9px;margin-top:54px}
.mosaico figure{aspect-ratio:1;overflow:hidden;border-radius:var(--raio);background:var(--suave)}
.mosaico figure:first-child{grid-column:span 2;grid-row:span 2}
.mosaico img{width:100%;height:100%;object-fit:cover;transition:transform .6s cubic-bezier(.2,.7,.3,1)}
.mosaico figure:hover img{transform:scale(1.04)}

/* servicos */
.servicos{background:var(--tinta);color:var(--fundo);margin-top:92px;padding:84px 0}
.servicos .rotulo{color:var(--destaque-claro)}
.servicos h2{font-family:var(--tec);font-size:clamp(1.5rem,3vw,2.1rem);
  letter-spacing:-.03em;font-weight:700;margin-top:10px}
.servicos ol{list-style:none;display:grid;
  grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:38px;margin-top:48px}
.conta{display:block;font-family:var(--tec);font-size:.78rem;font-weight:700;letter-spacing:.1em;
  color:var(--destaque-claro);padding-bottom:13px;border-bottom:1px solid rgba(255,255,255,.16)}
.servicos h3{font-family:var(--tec);font-size:1.12rem;font-weight:700;
  letter-spacing:-.02em;margin:15px 0 7px}
.servicos li p{opacity:.74;font-size:.95rem}
.sobre{margin-top:20px;max-width:56ch;opacity:.82;font-size:1.02rem}
.aconf{display:inline-block;margin-left:9px;padding:2px 9px;border-radius:100px;
  font-size:.6rem;font-weight:600;letter-spacing:.07em;text-transform:uppercase;
  background:var(--suave);color:var(--sutil);vertical-align:middle}

/* visita */
.visita{margin-top:92px;padding-bottom:104px;display:grid;
  grid-template-columns:1fr 1fr;gap:56px;align-items:start}
.visita h2{font-family:var(--tec);font-size:clamp(1.6rem,3.2vw,2.3rem);
  letter-spacing:-.03em;font-weight:700;margin-top:10px}
.dados{margin-top:26px;border-top:1px solid var(--linha)}
.dados>div{padding:17px 0;border-bottom:1px solid var(--linha)}
.dados dt{font-size:.68rem;font-weight:600;letter-spacing:.1em;
  text-transform:uppercase;color:var(--sutil)}
.dados dd{margin-top:5px;font-weight:500;font-size:1.02rem}
.botoes{display:flex;gap:11px;flex-wrap:wrap;margin-top:30px}
.btn{display:inline-flex;align-items:center;justify-content:center;padding:14px 27px;
  border-radius:100px;text-decoration:none;font-weight:600;font-size:.92rem;transition:.18s}
.btn-cheio{background:var(--destaque);color:#fff}
.btn-cheio:hover{filter:brightness(1.09);transform:translateY(-1px)}
.btn-vazio{border:1.5px solid var(--linha)}
.btn-vazio:hover{border-color:var(--tinta)}
.dupla{display:grid;grid-template-columns:1fr 1fr;gap:9px}
.dupla figure{aspect-ratio:3/4;overflow:hidden;border-radius:var(--raio);background:var(--suave)}
.dupla img{width:100%;height:100%;object-fit:cover}

/* rodape */
footer{border-top:1px solid var(--linha);padding:28px 0 32px}
footer .env{display:flex;justify-content:space-between;gap:14px;flex-wrap:wrap;
  color:var(--sutil);font-size:.86rem}

/* barra do celular */
.barra{position:fixed;left:0;right:0;bottom:0;z-index:50;display:none;gap:9px;padding:10px 12px;
  background:rgba(255,255,255,.95);backdrop-filter:blur(12px);border-top:1px solid var(--linha)}
.barra a{flex:1;text-align:center;padding:14px;border-radius:100px;
  text-decoration:none;font-weight:600;font-size:.92rem}

@media(max-width:860px){
  .visita{grid-template-columns:1fr;gap:34px}
  .mosaico{grid-template-columns:repeat(2,1fr)}
  .capa{aspect-ratio:4/3;max-height:none}
  .servicos,.visita{margin-top:70px}
  .link-tel{display:none}
  .barra{display:flex}
  body{padding-bottom:76px}
}
@media(prefers-reduced-motion:reduce){*{animation:none!important;transition:none!important}}
</style>
</head>
<body>

<header class="topo">
  <div class="env">
    <span class="marca">$nome</span>
    <div class="acoes">
      <a class="link-tel" href="$tel">$telefone</a>
      <a class="pilula" href="$acao_href">$acao_texto</a>
    </div>
  </div>
</header>

<img class="capa" src="$capa" alt="$nome">

<main>
  <section class="env abertura">
    <p class="rotulo">$rotulo</p>
    <h1>$nome</h1>
    <p class="onde">$bairro</p>
    <p class="selo"><span class="estrela">★</span> <b>$nota</b> no Google · $aval avaliações</p>

    <dl class="numeros">
      <div><dt>Nota</dt><dd>$nota</dd></div>
      <div><dt>Avaliações</dt><dd>$aval</dd></div>
      <div><dt>Categoria</dt><dd>$categoria</dd></div>
    </dl>
  </section>

  $bloco_mosaico

  <section class="servicos">
    <div class="env">
      <p class="rotulo">O que você encontra aqui</p>
      <h2>Do jeito que a gente trabalha</h2>
      <p class="sobre">$sobre</p>
      <ol>$cards</ol>
    </div>
  </section>

  <section class="env visita">
    <div>
      <p class="rotulo">Onde estamos</p>
      <h2>Venha até a gente</h2>
      <dl class="dados">
        <div><dt>Endereço</dt><dd>$endereco</dd></div>
        <div><dt>Telefone</dt><dd>$telefone</dd></div>
        <div><dt>Horário$marca</dt><dd>$horario</dd></div>
        <div><dt>Pagamento$marca</dt><dd>$pagamento</dd></div>
        $bloco_insta
      </dl>
      <div class="botoes">
        <a class="btn btn-cheio" href="$acao_href">$acao_texto</a>
        <a class="btn btn-vazio" href="$maps" target="_blank" rel="noopener">Ver no mapa</a>
      </div>
    </div>
    $bloco_dupla
  </section>
</main>

<footer>
  <div class="env"><span>$nome</span><span>$cidade</span></div>
</footer>

<nav class="barra">
  <a href="$tel" style="background:var(--suave);color:var(--tinta)">Ligar</a>
  <a href="$acao_href" style="background:var(--destaque);color:#fff">$acao_texto</a>
</nav>

</body>
</html>""")

# ─────────── gerar tudo ───────────
raiz = base / "sites"
if raiz.exists(): shutil.rmtree(raiz)
raiz.mkdir(parents=True)

acervo = []
for l in leads:
    if not l["pasta_fotos"]: continue
    fam = familia(l["ramo_busca"] + " " + l["categoria_google"])
    slug = sl(l["nome"])
    pasta = raiz / slug
    (pasta / "fotos").mkdir(parents=True, exist_ok=True)

    fotos = sorted(Path(l["pasta_fotos"]).glob("*.jpg"))
    for f in fotos: shutil.copy(f, pasta / "fotos" / f.name)
    if not fotos: continue
    nomes = [f"fotos/{f.name}" for f in fotos]

    destaque, origem = paleta_das_fotos([pasta / "fotos" / f.name for f in fotos], NICHO[fam])
    cor = derivar(destaque)

    nome_seguro = esc(l["nome"], quote=True)

    def figura(caminho):
        return f'<figure><img src="{caminho}" alt="{nome_seguro}" loading="lazy"></figure>'

    bloco_mosaico = ""
    if len(nomes) > 2:
        bloco_mosaico = ('<section class="env"><div class="mosaico">'
                         + "".join(figura(n) for n in nomes[1:6]) + "</div></section>")

    par = nomes[6:8] or nomes[1:3]
    bloco_dupla = f'<div class="dupla">{"".join(figura(n) for n in par)}</div>' if par else "<div></div>"

    cards = "".join(
        f'<li><span class="conta">{i:02d}</span><h3>{esc(t)}</h3><p>{esc(d)}</p></li>'
        for i, (t, d) in enumerate(SERVICOS[fam], 1)
    )

    tel_href, zap = contatos(l["telefone"])
    nota_br = str(l["nota"]).replace(".", ",")
    partes = [p.strip() for p in l["endereco"].split(",")]

    html = SITE.substitute(
        nome=nome_seguro, cidade=esc(CONFIG["cidade"]),
        nota=nota_br, aval=l["avaliacoes"],
        categoria=esc(l["categoria_google"] or l["ramo_busca"] or "Comércio local"),
        rotulo=esc((l["categoria_google"] or l["ramo_busca"]).upper()),
        endereco=esc(l["endereco"]),
        bairro=esc(", ".join(partes[1:3]) if len(partes) > 2 else l["endereco"]),
        telefone=esc(l["telefone"] or "-"), tel=tel_href or "#",
        maps=l["maps"] or "#", capa=nomes[0],
        cards=cards, bloco_mosaico=bloco_mosaico, bloco_dupla=bloco_dupla,
        bloco_insta=f'<div><dt>Instagram</dt><dd>{esc(l["instagram"])}</dd></div>' if l["instagram"] else "",
        sobre=esc(RASCUNHO[fam][0]),
        horario=esc(RASCUNHO[fam][1]),
        pagamento=esc(PAGAMENTO),
        marca='<span class="aconf">a confirmar</span>' if MARCAR_A_CONFIRMAR else "",
        acao_href=zap or tel_href or "#",
        acao_texto="Chamar no WhatsApp" if zap else "Ligar agora",
        **cor,
    )

    (pasta / "index.html").write_text(html, encoding="utf-8")

    tel_num = re.sub(r"\D", "", l["telefone"] or "")
    if tel_num and not tel_num.startswith("55"): tel_num = "55" + tel_num
    acervo.append({
        "slug": slug, "nome": l["nome"], "id": l["place_id"],
        "ramo": l["categoria_google"] or l["ramo_busca"],
        "nota": l["nota"], "aval": l["avaliacoes"], "km": l["km_do_centro"],
        "tel": l["telefone"] or "", "telnum": tel_num,
        "zap": tel_num if (len(tel_num) == 13 and tel_num[4] == "9") else "",
        "insta": l["instagram"], "end": l["endereco"], "maps": l["maps"],
        "capa": f"sites/{slug}/{nomes[0]}", "fotos": len(fotos),
        "cor": cor["destaque"], "origem": origem,
        "html": html.replace('src="fotos/', f'src="sites/{slug}/fotos/'),
    })
    print(f"  {l['nome']:<34} {cor['destaque']}  ({origem})")

print(f"\n{len(acervo)} sites gerados.")
print(f"{sum(1 for a in acervo if a['origem'] == 'foto')} com cor tirada das fotos, "
      f"{sum(1 for a in acervo if a['origem'] != 'foto')} com cor do nicho.")
if MARCAR_A_CONFIRMAR:
    print("\nHorário, pagamento e apresentação vieram preenchidos com texto genérico,")
    print("marcados com 'a confirmar'. Depois de fechar, peça os dados reais ao dono,")
    print("troque no estúdio (célula 7) e desligue MARCAR_A_CONFIRMAR aqui em cima.")
else:
    print("\nSem a marca 'a confirmar' — confira se os textos já são os reais.\n")

In [ ]:
#@title 📋 Painel de prospecção (rode depois de gerar os sites)

import json, re, unicodedata
from pathlib import Path

def sl(t):
    t = unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
    return (re.sub(r"[^a-zA-Z0-9]+", "-", t).strip("-").lower())[:50] or "site"

por_id = {a["id"]: a for a in acervo}

dados = []
for l in leads:
    a = por_id.get(l["place_id"])
    thumb = ""
    if l["pasta_fotos"]:
        p = Path(l["pasta_fotos"])
        fotos = sorted(p.glob("*.jpg"))
        if fotos:
            thumb = str(Path(l["pasta_fotos"]).relative_to(base) / fotos[0].name).replace("\\", "/")
    site = f"sites/{a['slug']}/index.html" if a else ""
    tel = re.sub(r"\D", "", l["telefone"] or "")
    if tel and not tel.startswith("55"):
        tel = "55" + tel
    dados.append({
        "nome": l["nome"], "tel": l["telefone"] or "", "telnum": tel,
        "zap": tel if (len(tel) == 13 and tel[4] == "9") else "",
        "nota": l["nota"], "aval": l["avaliacoes"], "km": l["km_do_centro"],
        "insta": l["instagram"], "end": l["endereco"], "maps": l["maps"],
        "ramo": l["categoria_google"] or l["ramo_busca"],
        "fotos": l["qtd_fotos"], "thumb": thumb, "site": site,
        "id": l["place_id"],
        "cor": a["cor"] if a else "",
        "origem": a["origem"] if a else "",
    })

HTML = """<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Prospeccao — __CIDADE__</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{
 --fundo:#ECEDE8; --carta:#fff; --tinta:#161A18; --sutil:#6E756F;
 --linha:#D7DAD3; --feito:#185C4A; --anda:#A8760A; --frio:#9AA09B;
 --tec:"Space Grotesk",system-ui,sans-serif; --txt:"Inter",system-ui,sans-serif;
}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--fundo);color:var(--tinta);font-family:var(--txt);line-height:1.5}
.env{max-width:1280px;margin:0 auto;padding:0 22px}

header{border-bottom:1px solid var(--linha);padding:26px 0 0;position:sticky;top:0;background:var(--fundo);z-index:10}
.titulo{display:flex;justify-content:space-between;align-items:baseline;gap:18px;flex-wrap:wrap}
h1{font-family:var(--tec);font-size:1.45rem;font-weight:700;letter-spacing:-.02em}
.contexto{font-size:.84rem;color:var(--sutil)}

.trilho{display:flex;height:6px;border-radius:3px;overflow:hidden;margin:18px 0 10px;background:var(--linha)}
.trilho div{transition:width .4s cubic-bezier(.2,.7,.3,1)}
.legenda{display:flex;gap:20px;font-size:.78rem;color:var(--sutil);padding-bottom:16px;flex-wrap:wrap}
.legenda b{font-family:var(--tec);color:var(--tinta)}
.ponto{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:6px}

.controles{display:flex;gap:9px;padding:14px 0 20px;flex-wrap:wrap;align-items:center}
input[type=search],select{font-family:var(--txt);font-size:.88rem;padding:10px 14px;border:1px solid var(--linha);
 background:var(--carta);border-radius:7px;color:var(--tinta)}
input[type=search]{flex:1;min-width:190px}
input:focus-visible,select:focus-visible,button:focus-visible,a:focus-visible{outline:2px solid var(--feito);outline-offset:2px}
.chip{font-family:var(--txt);font-size:.8rem;font-weight:600;padding:9px 15px;border:1px solid var(--linha);
 background:var(--carta);border-radius:100px;cursor:pointer;transition:.15s}
.chip[aria-pressed=true]{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}

.grade{display:grid;grid-template-columns:repeat(auto-fill,minmax(310px,1fr));gap:14px;padding-bottom:70px}
.carta{background:var(--carta);border:1px solid var(--linha);border-left:4px solid var(--frio);
 border-radius:9px;overflow:hidden;display:flex;flex-direction:column;transition:.18s}
.carta:hover{transform:translateY(-2px);box-shadow:0 8px 24px rgba(0,0,0,.07)}
.carta[data-s=anda]{border-left-color:var(--anda)}
.carta[data-s=feito]{border-left-color:var(--feito)}
.foto{aspect-ratio:16/9;background:var(--linha);position:relative}
.foto img{width:100%;height:100%;object-fit:cover;display:block}
.qtd{position:absolute;right:9px;bottom:9px;background:rgba(0,0,0,.72);color:#fff;font-family:var(--tec);
 font-size:.68rem;font-weight:500;padding:3px 8px;border-radius:4px}
.corpo{padding:15px 16px 16px;display:flex;flex-direction:column;gap:11px;flex:1}
.nome{font-family:var(--tec);font-size:1.02rem;font-weight:700;letter-spacing:-.015em;line-height:1.25}
.ramo{font-size:.72rem;color:var(--sutil);text-transform:uppercase;letter-spacing:.07em;margin-top:3px}
.metricas{display:flex;gap:16px;align-items:baseline;padding:9px 0;border-top:1px solid var(--linha);border-bottom:1px solid var(--linha)}
.met{font-family:var(--tec);font-size:1.12rem;font-weight:700}
.met small{display:block;font-family:var(--txt);font-size:.66rem;font-weight:500;color:var(--sutil);
 text-transform:uppercase;letter-spacing:.07em}
.cor{display:flex;align-items:center;gap:6px;font-size:.71rem;color:var(--sutil);margin-top:5px}
.cor i{width:11px;height:11px;border-radius:3px;flex-shrink:0;border:1px solid rgba(0,0,0,.12)}
.dados{font-size:.83rem;color:var(--sutil);display:flex;flex-direction:column;gap:3px}
.dados b{color:var(--tinta);font-weight:600}
.acoes{display:flex;gap:6px;flex-wrap:wrap;margin-top:auto}
.bt{font-family:var(--txt);font-size:.79rem;font-weight:600;padding:8px 13px;border-radius:6px;
 text-decoration:none;border:1px solid var(--linha);background:var(--carta);color:var(--tinta);cursor:pointer;transition:.15s}
.bt:hover{background:var(--fundo)}
.bt.forte{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}
.status{display:flex;gap:5px;margin-top:9px}
.status button{flex:1;font-size:.72rem;font-weight:600;padding:7px;border-radius:5px;border:1px solid var(--linha);
 background:var(--carta);cursor:pointer;color:var(--sutil)}
.status button[aria-pressed=true]{background:var(--tinta);color:var(--fundo);border-color:var(--tinta)}
textarea{width:100%;font-family:var(--txt);font-size:.82rem;padding:9px;border:1px solid var(--linha);
 border-radius:6px;resize:vertical;min-height:38px;background:var(--fundo);color:var(--tinta)}
.vazio{text-align:center;padding:70px 20px;color:var(--sutil)}
@media(prefers-reduced-motion:reduce){*{transition:none!important}}
</style></head>
<body>
<header><div class="env">
 <div class="titulo">
   <h1>Prospeccao __CIDADE__</h1>
   <span class="contexto">__TOTAL__ comercios sem site · raio de __RAIO__ km</span>
 </div>
 <div class="trilho" id="trilho"></div>
 <div class="legenda" id="legenda"></div>
 <div class="controles">
   <input type="search" id="busca" placeholder="Buscar por nome, bairro ou rua">
   <select id="ordem">
     <option value="aval">Mais avaliacoes</option>
     <option value="nota">Melhor nota</option>
     <option value="km">Mais perto</option>
     <option value="nome">Nome A-Z</option>
   </select>
   <button class="chip" id="fInsta" aria-pressed="false">Com Instagram</button>
   <button class="chip" id="fAbertos" aria-pressed="false">So a fazer</button>
   <button class="bt" id="exportar">Exportar CSV</button>
 </div>
</div></header>

<main class="env"><div class="grade" id="grade"></div></main>

<script>
const LEADS = __DADOS__;
const chave = "prospector:" + "__SLUG__";
let estado = JSON.parse(localStorage.getItem(chave) || "{}");
const salvar = () => localStorage.setItem(chave, JSON.stringify(estado));
const est = id => estado[id] || {s:"frio", nota:""};

const rotulos = {frio:["A fazer","var(--frio)"], anda:["Contatado","var(--anda)"], feito:["Fechado","var(--feito)"]};

function barra(){
  const c = {frio:0, anda:0, feito:0};
  LEADS.forEach(l => c[est(l.id).s]++);
  const t = LEADS.length || 1;
  document.getElementById("trilho").innerHTML =
    ["feito","anda","frio"].map(k => `<div style="width:${c[k]/t*100}%;background:${rotulos[k][1]}"></div>`).join("");
  document.getElementById("legenda").innerHTML =
    ["frio","anda","feito"].map(k =>
      `<span><i class="ponto" style="background:${rotulos[k][1]}"></i>${rotulos[k][0]} <b>${c[k]}</b></span>`).join("");
}

function desenhar(){
  const q = document.getElementById("busca").value.toLowerCase().trim();
  const ordem = document.getElementById("ordem").value;
  const soInsta = document.getElementById("fInsta").getAttribute("aria-pressed") === "true";
  const soAbertos = document.getElementById("fAbertos").getAttribute("aria-pressed") === "true";

  let lista = LEADS.filter(l => {
    if (q && !(l.nome + " " + l.end).toLowerCase().includes(q)) return false;
    if (soInsta && !l.insta) return false;
    if (soAbertos && est(l.id).s !== "frio") return false;
    return true;
  });

  lista.sort((a,b) => ordem === "nota" ? b.nota - a.nota
    : ordem === "km" ? a.km - b.km
    : ordem === "nome" ? a.nome.localeCompare(b.nome)
    : b.aval - a.aval);

  const g = document.getElementById("grade");
  if (!lista.length){ g.innerHTML = '<p class="vazio">Nenhum lead com esses filtros.</p>'; barra(); return; }

  g.innerHTML = lista.map(l => {
    const e = est(l.id);
    const foto = l.thumb
      ? `<div class="foto"><img src="${l.thumb}" alt="" loading="lazy"><span class="qtd">${l.fotos} fotos</span></div>`
      : `<div class="foto"></div>`;
    const zap = l.zap ? `<a class="bt" href="https://wa.me/${l.zap}" target="_blank" rel="noopener">WhatsApp</a>` : "";
    const insta = l.insta ? `<div>Instagram <b>${l.insta}</b></div>` : "";
    return `<article class="carta" data-s="${e.s}">
      ${foto}
      <div class="corpo">
        <div>
          <div class="nome">${l.nome}</div>
          <div class="ramo">${l.ramo}</div>
          ${l.cor ? `<div class="cor"><i style="background:${l.cor}"></i>cor ${l.origem === "foto" ? "tirada da foto" : "padrao do nicho"}</div>` : ""}
        </div>
        <div class="metricas">
          <div class="met">${String(l.nota).replace(".",",")}<small>Nota</small></div>
          <div class="met">${l.aval}<small>Avaliacoes</small></div>
          <div class="met">${String(l.km).replace(".",",")}<small>km</small></div>
        </div>
        <div class="dados">
          <div><b>${l.tel || "sem telefone"}</b></div>
          ${insta}
          <div>${l.end}</div>
        </div>
        <div class="acoes">
          ${l.site ? `<a class="bt forte" href="${l.site}" target="_blank" rel="noopener">Ver o site</a>`
                   : `<span class="bt" style="opacity:.45">Sem fotos</span>`}
          ${l.telnum ? `<a class="bt" href="tel:+${l.telnum}">Ligar</a>` : ""}
          ${zap}
          <a class="bt" href="${l.maps}" target="_blank" rel="noopener">Mapa</a>
        </div>
        <div class="status" data-id="${l.id}">
          ${["frio","anda","feito"].map(k =>
            `<button data-k="${k}" aria-pressed="${e.s===k}">${rotulos[k][0]}</button>`).join("")}
        </div>
        <textarea data-nota="${l.id}" placeholder="Anotacoes da visita">${e.nota||""}</textarea>
      </div>
    </article>`;
  }).join("");
  barra();
}

document.addEventListener("click", ev => {
  const b = ev.target.closest(".status button");
  if (b){
    const id = b.parentElement.dataset.id;
    estado[id] = {...est(id), s: b.dataset.k};
    salvar(); desenhar(); return;
  }
  const c = ev.target.closest(".chip");
  if (c){ c.setAttribute("aria-pressed", c.getAttribute("aria-pressed") !== "true"); desenhar(); }
});
document.addEventListener("input", ev => {
  if (ev.target.dataset.nota){
    const id = ev.target.dataset.nota;
    estado[id] = {...est(id), nota: ev.target.value};
    salvar();
  }
});
document.getElementById("busca").addEventListener("input", desenhar);
document.getElementById("ordem").addEventListener("change", desenhar);
document.getElementById("exportar").addEventListener("click", () => {
  const linhas = [["Nome","Telefone","Endereco","Nota","Avaliacoes","Km","Instagram","Status","Anotacoes"]];
  LEADS.forEach(l => {
    const e = est(l.id);
    linhas.push([l.nome,l.tel,l.end,String(l.nota).replace(".",","),l.aval,
      String(l.km).replace(".",","),l.insta,rotulos[e.s][0],(e.nota||"").replace(/\\n/g," ")]);
  });
  const csv = "\\uFEFF" + linhas.map(r => r.map(c => `"${String(c).replace(/"/g,'""')}"`).join(";")).join("\\n");
  const a = document.createElement("a");
  a.href = URL.createObjectURL(new Blob([csv], {type:"text/csv"}));
  a.download = "prospeccao.csv"; a.click();
});

desenhar();
</script>
</body></html>"""

html = (HTML
        .replace("__DADOS__", json.dumps(dados, ensure_ascii=False))
        .replace("__CIDADE__", CONFIG["cidade"])
        .replace("__TOTAL__", str(len(dados)))
        .replace("__RAIO__", str(int(CONFIG["raio_km"])))
        .replace("__SLUG__", sl(CONFIG["cidade"])))

(base / "painel.html").write_text(html, encoding="utf-8")

print(f"Painel gerado com {len(dados)} leads.")
print(f"{sum(1 for d in dados if d['site'])} com site pronto, "
      f"{sum(1 for d in dados if not d['site'])} sem fotos para montar site.")
print("\nO .zip com tudo sai na última célula.")

In [ ]:
#@title 🤖 Estúdio: editar os sites com o Claude

# A chave da Anthropic NAO fica escrita aqui nem dentro do HTML gerado.
# Voce cola ela uma vez dentro da propria pagina, e ela fica guardada so
# no SEU navegador. Assim o .zip pode ir para o cliente sem levar a chave.

import json, shutil
from google.colab import files

SISTEMA = """<!DOCTYPE html><html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Prospector</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@500;700&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<style>
:root{--fundo:#ECEDE8;--carta:#fff;--tinta:#161A18;--sutil:#6E756F;--linha:#D7DAD3;
--ok:#185C4A;--anda:#A8760A;--frio:#9AA09B;--tec:"Space Grotesk",sans-serif;--txt:"Inter",sans-serif}
*{box-sizing:border-box;margin:0;padding:0}
body{background:var(--fundo);color:var(--tinta);font-family:var(--txt);height:100vh;display:flex;overflow:hidden;font-size:14px}
aside{width:300px;background:var(--carta);border-right:1px solid var(--linha);display:flex;flex-direction:column;flex-shrink:0}
.cab{padding:16px 15px 12px;border-bottom:1px solid var(--linha)}
.cab h1{font-family:var(--tec);font-size:1.05rem;font-weight:700}
.cab p{font-size:.76rem;color:var(--sutil);margin-top:2px}
.trilho{display:flex;height:5px;border-radius:3px;overflow:hidden;margin:11px 0 7px;background:var(--linha)}
.leg{display:flex;gap:12px;font-size:.71rem;color:var(--sutil)}
.leg b{color:var(--tinta)}
.filtros{padding:10px 15px;border-bottom:1px solid var(--linha);display:flex;gap:6px;flex-wrap:wrap}
input,select{font-family:var(--txt);font-size:.82rem;padding:8px 11px;border:1px solid var(--linha);border-radius:6px;background:var(--carta);color:var(--tinta)}
#busca{width:100%}
.chip{font-size:.75rem;font-weight:600;padding:6px 12px;border:1px solid var(--linha);background:var(--carta);border-radius:100px;cursor:pointer}
.chip[aria-pressed=true]{background:var(--tinta);color:var(--carta);border-color:var(--tinta)}
#lista{flex:1;overflow-y:auto}
.item{padding:11px 15px;border-bottom:1px solid var(--linha);border-left:4px solid var(--frio);cursor:pointer}
.item:hover{background:var(--fundo)}
.item[aria-current=true]{background:var(--tinta);color:var(--carta)}
.item[data-s=anda]{border-left-color:var(--anda)}
.item[data-s=feito]{border-left-color:var(--ok)}
.item b{display:block;font-size:.86rem;font-weight:600;line-height:1.3}
.item span{font-size:.72rem;opacity:.6}
main{flex:1;display:flex;flex-direction:column;min-width:0}
.topo{padding:11px 18px;border-bottom:1px solid var(--linha);background:var(--carta);display:flex;
justify-content:space-between;align-items:center;gap:14px;flex-wrap:wrap}
.quem b{font-family:var(--tec);font-size:1rem;display:block}
.quem span{font-size:.76rem;color:var(--sutil)}
.bts{display:flex;gap:6px;flex-wrap:wrap}
button,.lk{font-family:var(--txt);font-size:.79rem;font-weight:600;padding:8px 13px;border-radius:6px;
border:1px solid var(--linha);background:var(--carta);color:var(--tinta);cursor:pointer;text-decoration:none;display:inline-block}
button:hover,.lk:hover{background:var(--fundo)}
button.ok{background:var(--ok);color:#fff;border-color:var(--ok)}
button:disabled{opacity:.4;cursor:not-allowed}
.status{display:flex;gap:4px}
.status button{padding:7px 11px;font-size:.74rem;color:var(--sutil)}
.status button[aria-pressed=true]{background:var(--tinta);color:var(--carta);border-color:var(--tinta)}
.palco{flex:1;background:#DEDFDA;display:flex;justify-content:center;overflow:hidden;padding:0}
iframe{border:0;background:#fff;width:100%;height:100%;transition:max-width .3s}
.chat{border-top:1px solid var(--linha);background:var(--carta);padding:10px 18px}
.hist{max-height:96px;overflow-y:auto;font-size:.78rem;color:var(--sutil);margin-bottom:8px}
.hist div{padding:2px 0}
.linha{display:flex;gap:7px}
.linha input{flex:1}
.vazio{flex:1;display:flex;align-items:center;justify-content:center;color:var(--sutil)}
</style></head><body>
<aside>
 <div class="cab"><h1>__CIDADE__</h1><p>__N__ comercios sem site</p>
  <div class="trilho" id="trilho"></div><div class="leg" id="leg"></div></div>
 <div class="filtros">
  <input id="busca" type="search" placeholder="Buscar nome ou rua">
  <button class="chip" id="fInsta" aria-pressed="false">Instagram</button>
  <button class="chip" id="fAbertos" aria-pressed="false">A fazer</button>
  <button class="chip" id="exportar">Exportar CSV</button>
  <button class="chip" id="btChave">Chave</button>
 </div>
 <div id="lista"></div>
</aside>
<main>
 <div class="topo">
  <div class="quem"><b id="tit">Escolha um comercio</b><span id="sub"></span></div>
  <div class="bts" id="acoes"></div>
 </div>
 <div class="palco"><iframe id="prev" title="Previa do site"></iframe></div>
 <div class="chat"><div class="hist" id="hist"></div>
  <div class="linha"><input id="pedido" placeholder="Ex: encurte o texto do primeiro bloco">
  <button id="enviar">Pedir a IA</button></div></div>
</main>
<script>
const SITES=__ACERVO__, CH="prospector:__SLUG__", CHK="prospector:chave-claude";
const MODELO="claude-opus-5";

function chave(){ try{ return localStorage.getItem(CHK)||""; }catch(e){ return ""; } }
function pedirChave(){
 const v=prompt("Cole a sua chave da Anthropic (comeca com sk-ant-).\n\n"
  +"Ela fica guardada so neste navegador. Nao entra no arquivo nem vai junto para o cliente.", chave());
 if(v===null) return chave();
 const c=v.trim();
 try{ c ? localStorage.setItem(CHK,c) : localStorage.removeItem(CHK); }catch(e){}
 return c;
}
const SISTEMA_IA=`Voce edita paginas HTML de sites de comercios de bairro brasileiros.

Devolva SEMPRE o arquivo HTML completo, do <!DOCTYPE html> ate </html>.
Nada antes, nada depois, sem cerca de codigo, sem explicacao.

Regras que nao se quebram:
- Mantenha os caminhos das imagens exatamente como estao.
- Mantenha os links de telefone, WhatsApp e mapa funcionando.
- Mantenha as variaveis de cor (--tinta, --fundo, --destaque, --suave, --tinta-rgb)
  como estao, a menos que o pedido seja justamente sobre cor.
- Nao invente servico, preco, horario, promocao nem promessa que nao esteja
  no HTML atual. Se faltar o dado, deixe o texto generico em vez de inventar.
- Portugues do Brasil, com acentos. O site tem que continuar funcionando no celular.
- Mude apenas o que foi pedido.`;

let estado=JSON.parse(localStorage.getItem(CH)||"{}"), atual=null, pilha=[];
const salvar=()=>localStorage.setItem(CH,JSON.stringify(estado));
const est=id=>estado[id]||{s:"frio",nota:""};
const ROT={frio:["A fazer","#9AA09B"],anda:["Contatado","#A8760A"],feito:["Aprovado","#185C4A"]};

function barra(){
 const c={frio:0,anda:0,feito:0}; SITES.forEach(s=>c[est(s.id).s]++);
 const t=SITES.length||1;
 trilho.innerHTML=["feito","anda","frio"].map(k=>`<div style="width:${c[k]/t*100}%;background:${ROT[k][1]}"></div>`).join("");
 leg.innerHTML=["frio","anda","feito"].map(k=>`<span>${ROT[k][0]} <b>${c[k]}</b></span>`).join("");
}

function lista_(){
 const q=busca.value.toLowerCase().trim();
 const si=fInsta.getAttribute("aria-pressed")==="true";
 const sa=fAbertos.getAttribute("aria-pressed")==="true";
 const vis=SITES.map((s,i)=>({s,i})).filter(({s})=>{
  if(q&&!(s.nome+" "+s.end).toLowerCase().includes(q))return false;
  if(si&&!s.insta)return false;
  if(sa&&est(s.id).s!=="frio")return false;
  return true;});
 document.getElementById("lista").innerHTML = vis.length? vis.map(({s,i})=>
  `<div class="item" data-i="${i}" data-s="${est(s.id).s}" aria-current="${i===atual}">
   <b>${s.nome}</b><span>${String(s.nota).replace(".",",")} · ${s.aval} avaliacoes · ${String(s.km).replace(".",",")} km</span></div>`
  ).join("") : '<p class="vazio" style="padding:30px;font-size:.85rem">Nada com esses filtros.</p>';
 barra();
}

function abrir(){
 const s=SITES[atual], e=est(s.id);
 prev.srcdoc=s.html;
 tit.textContent=s.nome;
 sub.textContent=`${s.tel||"sem telefone"} · ${s.ramo}`;
 acoes.innerHTML=`
  <div class="status" data-id="${s.id}">
   ${["frio","anda","feito"].map(k=>`<button data-k="${k}" aria-pressed="${e.s===k}">${ROT[k][0]}</button>`).join("")}
  </div>
  ${s.telnum?`<a class="lk" href="tel:+${s.telnum}">Ligar</a>`:""}
  ${s.zap?`<a class="lk" href="https://wa.me/${s.zap}" target="_blank" rel="noopener">WhatsApp</a>`:""}
  <a class="lk" href="${s.maps}" target="_blank" rel="noopener">Mapa</a>
  <button id="larg">Celular</button>
  <button id="desfazer" ${pilha.length?"":"disabled"}>Desfazer</button>
  <button id="baixar" class="ok">Aprovar e baixar</button>`;
 lista_();
}

document.addEventListener("click",ev=>{
 const it=ev.target.closest(".item");
 if(it){ atual=+it.dataset.i; pilha=[]; hist.innerHTML=""; abrir(); return; }
 const st=ev.target.closest(".status button");
 if(st){ const id=st.parentElement.dataset.id;
  estado[id]={...est(id),s:st.dataset.k}; salvar(); abrir(); return; }
 const ch=ev.target.closest(".chip");
 if(ch&&ch.id!=="exportar"){ ch.setAttribute("aria-pressed",ch.getAttribute("aria-pressed")!=="true"); lista_(); }
 if(ev.target.id==="desfazer"&&pilha.length){ SITES[atual].html=pilha.pop(); abrir(); log("desfeito"); }
 if(ev.target.id==="larg"){ const f=prev.style.maxWidth==="420px";
  prev.style.maxWidth=f?"none":"420px"; ev.target.textContent=f?"Celular":"Computador"; }
 if(ev.target.id==="baixar"){ const s=SITES[atual];
  const saida=s.html.replaceAll(`src="sites/${s.slug}/fotos/`,'src="fotos/');
  const a=document.createElement("a");
  a.href=URL.createObjectURL(new Blob([saida],{type:"text/html"}));
  a.download="index.html"; a.click();
  log("baixado — substitua em sites/"+s.slug+"/"); }
});

function log(t){ hist.innerHTML+=`<div>${t}</div>`; hist.scrollTop=hist.scrollHeight; }

enviar.addEventListener("click",async()=>{
 if(atual===null){ log("Escolha um comercio na lista."); return; }
 const p=pedido.value.trim();
 if(!p){ log("Escreva o que quer mudar."); return; }
 enviar.disabled=true; enviar.textContent="Editando...";
 log("→ "+p);
 let k=chave(); if(!k) k=pedirChave();
 if(!k){ log("Sem a chave nao da para editar."); enviar.disabled=false; enviar.textContent="Pedir a IA"; return; }
 const instr=`Mudanca pedida: ${p}

HTML atual do site:

${SITES[atual].html}`;
 try{
  const r=await fetch("https://api.anthropic.com/v1/messages",
   {method:"POST",
    headers:{"content-type":"application/json","x-api-key":k,
             "anthropic-version":"2023-06-01",
             "anthropic-dangerous-direct-browser-access":"true"},
    body:JSON.stringify({model:MODELO,max_tokens:32000,system:SISTEMA_IA,
                         messages:[{role:"user",content:instr}]})});
  const d=await r.json();
  if(!r.ok){
   if(r.status===401){ log("A chave nao foi aceita. Clique em 'Chave' e cole de novo."); }
   else if(r.status===429){ log("Muitos pedidos seguidos. Espere um minuto."); }
   else { log("Erro: "+(d.error?.message||r.status)); }
   return; }
  if(d.stop_reason==="refusal"){ log("O Claude preferiu nao fazer essa mudanca."); return; }
  if(d.stop_reason==="max_tokens"){ log("A resposta foi cortada. Peca uma mudanca menor por vez."); return; }
  let novo=(d.content||[]).filter(b=>b.type==="text").map(b=>b.text).join("")
            .replace(/^```html\\s*/i,"").replace(/```\\s*$/,"").trim();
  if(!novo.toLowerCase().includes("<!doctype")){ log("Resposta incompleta. Tente um pedido menor."); return; }
  pilha.push(SITES[atual].html); SITES[atual].html=novo; abrir(); log("aplicado"); pedido.value="";
 }catch(e){ log("Falhou: "+e.message); }
 finally{ enviar.disabled=false; enviar.textContent="Pedir a IA"; }
});

document.getElementById("btChave").addEventListener("click",()=>{
 const c=pedirChave();
 log(c ? "Chave guardada neste navegador." : "Chave apagada deste navegador.");
});
pedido.addEventListener("keydown",e=>{ if(e.key==="Enter") enviar.click(); });
busca.addEventListener("input",lista_);
exportar.addEventListener("click",()=>{
 const L=[["Nome","Telefone","Endereco","Nota","Avaliacoes","Km","Instagram","Status"]];
 SITES.forEach(s=>L.push([s.nome,s.tel,s.end,String(s.nota).replace(".",","),s.aval,
  String(s.km).replace(".",","),s.insta,ROT[est(s.id).s][0]]));
 const csv="\\uFEFF"+L.map(r=>r.map(c=>`"${String(c).replace(/"/g,'""')}"`).join(";")).join("\\n");
 const a=document.createElement("a");
 a.href=URL.createObjectURL(new Blob([csv],{type:"text/csv"}));
 a.download="prospeccao.csv"; a.click();
});
lista_();
</script></body></html>"""

(base / "sistema.html").write_text(
    SISTEMA.replace("__ACERVO__", json.dumps(acervo, ensure_ascii=False))
           .replace("__CIDADE__", CONFIG["cidade"])
           .replace("__N__", str(len(acervo)))
           .replace("__SLUG__", sl(CONFIG["cidade"])), encoding="utf-8")

print(f"sistema.html pronto com {len(acervo)} comércios.")
print("Abra o sistema.html, clique em 'Chave' e cole a sua chave da Anthropic.")
print("A chave fica só no seu navegador — o .zip pode ir para o cliente sem ela.")

In [ ]:
#@title 📥 Baixar tudo num arquivo só

import shutil
from pathlib import Path
from google.colab import files

nome = f"prospeccao-{sl(CONFIG['cidade'])}-{int(CONFIG['raio_km'])}km"
caminho = shutil.make_archive(nome, "zip", base)
tamanho = Path(caminho).stat().st_size / (1024 * 1024)

print(f"{nome}.zip — {tamanho:.1f} MB\n")
print("Dentro dele:")
print("  sistema.html       o estúdio: edite os sites conversando com o Claude")
print("  painel.html        a lista de prospecção, com status e anotações")
print("  leads.pdf          a lista para imprimir ou mandar por WhatsApp")
print("  leads_excel.csv    abre direto no Excel brasileiro")
print("  sites/             um site pronto por lead")
print("  fotos/             as fotos baixadas do Google\n")
print("Descompacte a pasta inteira e abra o sistema.html no navegador.")
print("Os links só funcionam com a pasta junto.\n")
print("A sua chave do Claude NÃO está neste arquivo — ela fica no seu navegador.")
print("Você pode mandar esta pasta para o cliente sem medo.\n")
print("⚠️  As fotos são de quem as enviou ao Google. Use para montar a proposta")
print("    e mostrar o preview. Só publique em site aberto depois do sim do dono.")

files.download(caminho)